# Indium Zinc Oxide Ceramic Composition and Thin Film Properties Measurements Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIRˆ2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.8mqj-hhz4/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.8mqj-hhz4/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

In [ ]:
# Show record sets and their fields
record_sets = metadata.recordSet
print("Available record sets:")
for rs in record_sets:
    print(f"  RecordSet: {rs['@id']}")
    # List fields within the record set
    if 'field' in rs:
        print("    Fields:")
        for field in rs['field']:
            print(f"      Field: {field['@id']} ({field.get('name','')})")
    # List columns within the record set
    if 'column' in rs:
        print("    Columns:")
        for col in rs['column']:
            print(f"      Column: {col['@id']} ({col.get('name','')})")
    print()

print("-- Example records from first record set --")
if record_sets:
    first_rs_id = record_sets[0]['@id']
    for i, x in enumerate(dataset.records(record_set=first_rs_id)):
        if i > 2: break
        print(x)

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Collect all record set @id's
record_set_ids = [rs['@id'] for rs in record_sets]
print("RecordSet IDs:", record_set_ids)

dataframes = {}
for rsid in record_set_ids:
    try:
        records = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f\n{rsid} columns:", df.columns.tolist())
        print(df.head())
    except Exception as e:
        print(f"Could not load records for {rsid}: {e}")

# Use first record set for further analysis, if available
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
else:
    selected_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
Operations include removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
# Check numeric columns for analysis in selected RecordSet DataFrame
df = dataframes.get(selected_record_set_id)
if df is not None:
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"Numeric fields in {selected_record_set_id}:", numeric_fields)

    # Select one numeric field, fallback if not found
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
    else:
        numeric_field_id = None
else:
    numeric_field_id = None

# Example filtering and normalization
if numeric_field_id:
    threshold = df[numeric_field_id].quantile(0.5)  # median threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f\nFiltered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by a key attribute (if exists)
    possible_group_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
    if possible_group_fields:
        group_field_id = possible_group_fields[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f\nGrouped data by {group_field_id}:")
        print(grouped_df.head())
    else:
        group_field_id = None
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
# Visualization examples
if df is not None and numeric_field_id:
    plt.figure(figsize=(7, 4))
    plt.hist(df[numeric_field_id].dropna(), bins=30, color='slateblue', alpha=0.7)
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.title(f"Distribution of {numeric_field_id} in {selected_record_set_id}")
    plt.show()

    # Boxplot by group field (if available)
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(8, 4))
        df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded the Indium Zinc Oxide Ceramic dataset using `mlcroissant` from the Croissant schema URL.
- Identified available record sets, fields, and columns by their `@id`s.
- Extracted tabular data and performed basic cleaning, filtering, and normalization.
- Visualized numeric field distributions and relationships by group (if present).

**This provides a reproducible workflow for material datasets to support structure–property analysis and further ML applications.**